Business Question #1: Margin/Revenue anomally
What drove the anomalous swings in June (margin/revenue collapse followed by an AOV spike), and was this driven by a sudden shift in channel mix or category demand?

In [45]:
with monthly as (
SELECT DATENAME (month, CAST(orderdate as DATE)) AS Month, 
    DATEPART(MONTH,orderdate) as month_num,
    round(sum(ExtendedAmount),2) as Revenue,
    SUM(ExtendedAmount) - SUM (totalproductcost) as Profit,
    (SUM(ExtendedAmount ) - SUM (totalproductcost)) / SUM(ExtendedAmount ) AS Margin
FROM FactInternetSales
GROUP BY DATEPART(MONTH,orderdate),DATENAME (month, CAST(orderdate as DATE))

)

SELECT Month,Revenue, 
    ROUND((revenue - lag (revenue, 1) OVER (ORDER BY month_num)) / 
    lag (revenue, 1) OVER (ORDER BY month_num) * 100,2) AS MoM_Revenue
from monthly
ORDER BY month_num

(12 rows affected)

Month     | Revenue   | MoM_Revenue
----------+-----------+------------
January   | 42464.61  | NULL       
February  | 34479.15  | -18.8      
March     | 44131.69  | 28         
April     | 60123.42  | 36.24      
May       | 66601.78  | 10.78      
June      | 72611.31  | 9.02       
July      | 101458.07 | 39.73      
August    | 71331.39  | -29.69     
September | 58265.88  | -18.32     
October   | 80042.06  | 37.37      
November  | 68296.15  | -14.67     
December  | 87089.06  | 27.52      
(12 rows)

Total execution time: 00:00:00.190

As we dive into our analysis we confirm the sudden drop collapse of revenue in june,is not present in our B2C channel so it  should be concentrated in the Reseller channel(B2B) and not our B2C channel.

In [44]:
with monthly as (
SELECT DATENAME (month, CAST(orderdate as DATE)) AS Month, 
    DATEPART(MONTH,orderdate) as month_num,
    round(sum(ExtendedAmount),2) as Revenue,
    SUM(ExtendedAmount) - SUM (totalproductcost) as Profit,
    (SUM(ExtendedAmount ) - SUM (totalproductcost)) / SUM(ExtendedAmount ) AS Margin
FROM factresellersales
GROUP BY DATEPART(MONTH,orderdate),DATENAME (month, CAST(orderdate as DATE))

)

SELECT Month,Revenue, 
    ROUND((revenue - lag (revenue, 1) OVER (ORDER BY month_num)) / 
    lag (revenue, 1) OVER (ORDER BY month_num) * 100,2) AS MoM_Revenue
from monthly
ORDER BY month_num

(12 rows affected)

Month     | Revenue   | MoM_Revenue
----------+-----------+------------
January   | 225289.38 | NULL       
February  | 207087.78 | -8.08      
March     | 168056.08 | -18.85     
April     | 137318.55 | -18.29     
May       | 278950.32 | 103.14     
June      | 71741.48  | -74.28     
July      | 123067.5  | 71.54      
August    | 178305.23 | 44.88      
September | 114748.16 | -35.65     
October   | 166962.33 | 45.5       
November  | 196300.93 | 17.57      
December  | 142031.81 | -27.65     
(12 rows)

Total execution time: 00:00:00.193

In [62]:
WITH segment AS (
    SELECT 
        EnglishProductSubcategoryName AS Category,
        DATENAME(month, CAST(fs.OrderDate AS DATE)) AS MonthName,
        ROUND(SUM(fs.ExtendedAmount), 2) AS Revenue,
        COUNT(DISTINCT fs.SalesOrderNumber) AS OrderCount
    FROM FactResellerSales fs
    LEFT JOIN DimProduct dp ON dp.ProductKey = fs.ProductKey
    LEFT JOIN DimProductSubcategory dps ON dp.ProductSubcategoryKey = dps.ProductSubcategoryKey
    GROUP BY EnglishProductSubcategoryName, DATENAME(month, CAST(fs.OrderDate AS DATE))
),
baseline AS (
    SELECT 
        Category,
        AVG(OrderCount) AS TypicalOrders,
        ROUND(AVG(Revenue), 2) AS TypicalRevenue
    FROM segment
    WHERE LOWER(MonthName) <> 'june'
    GROUP BY Category
),
june AS (
    SELECT Category, Revenue AS JuneRevenue, OrderCount AS JuneOrders
    FROM segment
    WHERE LOWER(MonthName) = 'june'
)
SELECT 
    j.Category,
    j.JuneRevenue,
    j.JuneOrders,
    b.TypicalRevenue,
    b.TypicalOrders,
    ROUND(j.JuneRevenue - b.TypicalRevenue, 2) AS RevenueDeviation
FROM june j
JOIN baseline b ON j.Category = b.Category
ORDER BY ABS(j.JuneRevenue - b.TypicalRevenue) DESC;

(16 rows affected)

Category        | JuneRevenue | JuneOrders | TypicalRevenue | TypicalOrders | RevenueDeviation
----------------+-------------+------------+----------------+---------------+-----------------
Mountain Bikes  | 17184.01    | 4          | 54139.54       | 14            | -36955.53       
Road Bikes      | 37120.74    | 10         | 62338.88       | 21            | -25218.14       
Touring Bikes   | 7179.59     | 4          | 26182.01       | 8             | -19002.42       
Mountain Frames | 1495.9      | 3          | 12275.87       | 9             | -10779.97       
Road Frames     | 2359.04     | 2          | 8838.14        | 10            | -6479.1         
Touring Frames  | 3011.73     | 1          | 4033.53        | 3             | -1021.8         
Jerseys         | 443.17      | 4          | 1430.82        | 8             | -987.65         
Wheels          | 735.72      | 2          | 1535.34        | 4             | -799.62         
Helmets         | 184.1       

As we dive further into our analsysis, we notice that the drop in sales is concentrated in bikes subcategories, mountain & road bikes accounting for the vast majority of the drop in that month

In [69]:
WITH segment AS (
    SELECT 
         EnglishCountryRegionName AS region,
        DATENAME(month, CAST(fs.OrderDate AS DATE)) AS MonthName,
        ROUND(SUM(fs.ExtendedAmount), 2) AS Revenue,
        COUNT(DISTINCT fs.SalesOrderNumber) AS OrderCount
    FROM factinternetsales fs
    LEFT JOIN DimGeography dg ON dg.SalesTerritoryKey = fs.SalesTerritoryKey
    
    GROUP BY EnglishCountryRegionName, DATENAME(month, CAST(fs.OrderDate AS DATE))
),
baseline AS (
    SELECT 
        region,
        AVG(OrderCount) AS TypicalOrders,
        ROUND(AVG(Revenue), 2) AS TypicalRevenue
    FROM segment
    WHERE LOWER(MonthName) <> 'june'
    GROUP BY region
),
june AS (
    SELECT region, Revenue AS JuneRevenue, OrderCount AS JuneOrders
    FROM segment
    WHERE LOWER(MonthName) = 'june'
)
SELECT 
    j.region,
    j.JuneRevenue,
    j.JuneOrders,
    b.TypicalRevenue,
    b.TypicalOrders,
    ROUND(j.JuneRevenue - b.TypicalRevenue, 2) AS RevenueDeviation
FROM june j
JOIN baseline b ON j.region = b.region
ORDER BY ABS(j.JuneRevenue - b.TypicalRevenue) DESC;

(6 rows affected)

region         | JuneRevenue | JuneOrders | TypicalRevenue | TypicalOrders | RevenueDeviation
---------------+-------------+------------+----------------+---------------+-----------------
United States  | 3141219.96  | 47         | 2206484.64     | 42            | 934735.32       
United Kingdom | 218114.61   | 12         | 470767.34      | 13            | -252652.73      
Canada         | 493471.24   | 19         | 290218.85      | 15            | 203252.39       
France         | 106592.64   | 15         | 276967.72      | 10            | -170375.08      
Australia      | 882485.5    | 30         | 730027.29      | 28            | 152458.21       
Germany        | 549495.7    | 16         | 484908.65      | 10            | 64587.05        
(6 rows)

Total execution time: 00:00:00.946

As we deep further into our analysis we noticed that 2 regions, actually account for the majority of the revenue drop anomally  of the unexplained june drop. 
Executive Recommendation:

we strongly recommend the Sales Operations team audit reseller partnerships in the UK and France. The primary investigation points should be:

Contract Review: Were any large, recurring B2B wholesale contracts delayed, paused, or canceled in June?
Competitive Analysis: Did a competitor launch an aggressive regional promotion or undercut our wholesale pricing in Europe during this period?
Supply Chain Check: Were there specific logistical delays or stockouts at our European distribution centers that prevented resellers from placing orders?"

Business question #2   Financial Mechanics
Is the B2B vs. B2C margin gap primarily driven by wholesale pricing/discount structures, 
or is it driven by the specific product mix each channel sells?

In [101]:
SELECT englishproductname as product, 
    unitprice,
    productstandardcost,
    unitprice - productstandardcost AS profit_per_unit,
    COUNT(salesordernumber) Orders,
    sum(orderquantity) as Quantity,
    SUM(ExtendedAmount) as Revenue,
    SUM(ExtendedAmount) - SUM (totalproductcost) as Profit,
    SUM(DiscountAmount) as Discount,
    (SUM(ExtendedAmount ) - SUM (totalproductcost)) / SUM(ExtendedAmount ) AS Margin,
    SUM(TotalProductCost) as Cost,
    sum(DiscountAmount) / SUM(ExtendedAmount) * 100 AS disc_perc
FROM factinternetsales fi
LEFT JOIN DimProduct dp ON dp.ProductKey = fi.ProductKey
GROUP BY englishproductname,unitprice,productstandardcost
ORDER BY Revenue DESC


(135 rows affected)

product                         | unitprice | productstandardcost | profit_per_unit    | Orders | Quantity | Revenue            | Profit             | Discount | Margin              | Cost               | disc_perc
--------------------------------+-----------+---------------------+--------------------+--------+----------+--------------------+--------------------+----------+---------------------+--------------------+----------
Road-150 Red, 52                | 3578.27   | 2171.2942           | 1406.9758000000002 | 11     | 11       | 39360.969999999994 | 15476.733799999995 | 0        | 0.3932000100607276  | 23884.2362         | 0        
Mountain-200 Black, 46          | 2294.99   | 1251.9813           | 1043.0086999999999 | 16     | 16       | 36719.83999999998  | 16688.139199999987 | 0        | 0.4544720020566537  | 20031.700799999995 | 0        
Road-150 Red, 44                | 3578.27   | 2171.2942           | 1406.9758000000002 | 10     | 10       | 35782.7   

In [99]:
SELECT englishproductname as product, 
    unitprice,
    productstandardcost,
    unitprice - productstandardcost AS profit_per_unit,
    COUNT(salesordernumber) Orders,
    sum(orderquantity) as Quantity,
    SUM(ExtendedAmount) as Revenue,
    SUM(ExtendedAmount) - SUM (totalproductcost) as Profit,
    SUM(DiscountAmount) as Discount,
    (SUM(ExtendedAmount ) - SUM (totalproductcost)) / SUM(ExtendedAmount ) AS Margin,
    SUM(TotalProductCost) as Cost,
    sum(DiscountAmount) / SUM(ExtendedAmount) * 100 AS disc_perc

FROM factresellersales fr
LEFT JOIN DimProduct dp ON dp.ProductKey = fr.ProductKey
GROUP BY englishproductname,unitprice,productstandardcost
ORDER BY Revenue DESC


(352 rows affected)

product                          | unitprice | productstandardcost | profit_per_unit      | Orders | Quantity | Revenue            | Profit              | Discount           | Margin                | Cost               | disc_perc         
---------------------------------+-----------+---------------------+----------------------+--------+----------+--------------------+---------------------+--------------------+-----------------------+--------------------+-------------------
Mountain-200 Black, 42           | 1229.4589 | 1105.81             | 123.64890000000014   | 13     | 54       | 66390.78059999998  | 6677.040599999986   | 0                  | 0.10057180439297302   | 59713.74           | 0                 
Road-350-W Yellow, 48            | 1020.594  | 1082.51             | -61.91599999999994   | 14     | 49       | 50009.10600000001  | -3033.8839999999836 | 0                  | -0.06066663139309036  | 53042.98999999999  | 0                 
Road-250 Black, 48 

The products in b2b have the same cost per unit but a much lower unit price, This wholesale intrinsec  discount and not the promotional discount what's driving the gap in margin and profit between the two categories 
only 5 products had discount ratio to revenue above 30% and they didn't have signficant sales or revenue,most products dont have discount and the top 20 of products with the most revenue didn't have any discount at all except for one with a discount of 2%.

so promotional discount is not driving the margin gap, intrensic or wholesale discount is,
both channels are selling the same categories the most: bikes with road and mountain bike account for the most revenue so product mix is not driving the profit/margin gap wholesale discount is.